In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, f1_score

df = pd.read_parquet(r"..\datasets\merged.parquet")
df = df[df['ir_length'] == 1800].reset_index(drop=True)
print(f"Full dataset: {len(df)} samples")

Full dataset: 191153 samples


instead of taking this subset approach we can also go stratified sampling either proportionate or disproportionate  
make sure to use the interpolated data

In [2]:
# Keep all rare class samples + random sample of common ones
rare  = df[(df['sulfonic_acid'] == 1) | (df['guanidino'] == 1)]
common = df[(df['sulfonic_acid'] == 0) & (df['guanidino'] == 0)].sample(
    n=5000, random_state=42)

df_sub = pd.concat([rare, common]).sample(
    frac=1, random_state=42).reset_index(drop=True)
print(f"Subset size: {len(df_sub)} samples")
print(f"Label counts:")
for col in ['carboxylic_acid', 'amino', 'sulfonic_acid', 'guanidino']:
    print(f"  {col}: {int(df_sub[col].sum())}")

X = np.vstack(df_sub['ir_spectra'].values)
y = df_sub[['carboxylic_acid', 'amino', 'sulfonic_acid', 'guanidino']].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

Subset size: 8196 samples
Label counts:
  carboxylic_acid: 2372
  amino: 4786
  sulfonic_acid: 68
  guanidino: 3128

Train: (6556, 1800), Test: (1640, 1800)


In [3]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("\nTraining Random Forest...")
rf = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators  = 100,
        class_weight  = 'balanced',
        random_state  = 42,
        n_jobs        = -1
    )
)
rf.fit(X_train_sc, y_train)
print("Done")


Training Random Forest...
Done


In [4]:
y_pred = rf.predict(X_test_sc)

CLASS_NAMES = ['carboxylic_acid', 'amino', 'sulfonic_acid', 'guanidino']

print("\nRandom Forest Results:")
print(classification_report(
    y_test, y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
))

print(f"Macro F1: {f1_score(y_test, y_pred, average='macro', zero_division=0):.3f}")


Random Forest Results:
                 precision    recall  f1-score   support

carboxylic_acid       0.98      0.95      0.97       484
          amino       0.98      0.99      0.99       931
  sulfonic_acid       1.00      0.29      0.44        14
      guanidino       0.96      0.81      0.88       629

      micro avg       0.97      0.92      0.95      2058
      macro avg       0.98      0.76      0.82      2058
   weighted avg       0.97      0.92      0.95      2058
    samples avg       0.93      0.92      0.92      2058

Macro F1: 0.819


Trying out different configurations.  
Setting n_estimators to 300 instead of default 100.  
Look into if there is a way to optimize this number?  
class_weight is configured to be more flexible and usually better for rare classes.


In [8]:
rf2 = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators = 300,
        class_weight = 'balanced_subsample',
        random_state = 42,
        n_jobs = -1
    )
)
rf2.fit(X_train_sc, y_train)
y_pred2 = rf2.predict(X_test_sc)
print("RF (300 trees, balanced_subsample):")
print(classification_report(y_test, y_pred2,
    target_names=CLASS_NAMES, zero_division=0))

RF (300 trees, balanced_subsample):
                 precision    recall  f1-score   support

carboxylic_acid       0.98      0.95      0.97       484
          amino       0.98      1.00      0.99       931
  sulfonic_acid       1.00      0.29      0.44        14
      guanidino       0.95      0.81      0.88       629

      micro avg       0.97      0.93      0.95      2058
      macro avg       0.98      0.76      0.82      2058
   weighted avg       0.97      0.93      0.95      2058
    samples avg       0.93      0.93      0.92      2058



Lowering Thresholds
Default threshold is 0.5  
The model sees very few positive examples of rare classes so is less confident about them. Therefore lowering the threshold can help.
Result of lowering threshold globally:

In [ ]:
y_proba = np.array([est.predict_proba(X_test_sc)[:, 1]
                    for est in rf.estimators_]).T

threshold = 0.2
y_pred_low = (y_proba > threshold).astype(int)
print(f"\nRF with threshold={threshold} for all classes:")
print(classification_report(y_test, y_pred_low,
    target_names=CLASS_NAMES, zero_division=0))


RF with threshold=0.2 for all classes:
                 precision    recall  f1-score   support

carboxylic_acid       0.86      0.99      0.92       484
          amino       0.87      1.00      0.93       931
  sulfonic_acid       0.73      0.79      0.76        14
      guanidino       0.65      0.99      0.78       629

      micro avg       0.78      0.99      0.88      2058
      macro avg       0.78      0.94      0.85      2058
   weighted avg       0.80      0.99      0.88      2058
    samples avg       0.84      0.99      0.89      2058



Lower only for sulfonic acid

In [ ]:
# Use default threshold (0.5) for all classes except sulfonic acid
y_proba = np.array([est.predict_proba(X_test_sc)[:, 1]
                    for est in rf.estimators_]).T

thresholds = {
    'carboxylic_acid': 0.5,
    'amino':           0.5,
    'sulfonic_acid':   0.2,
    'guanidino':       0.5,
}

y_pred_targeted = np.column_stack([
    (y_proba[:, i] > list(thresholds.values())[i]).astype(int)
    for i in range(len(CLASS_NAMES))
])

print("RF with targeted threshold for sulfonic acid only:")
print(classification_report(y_test, y_pred_targeted,
    target_names=CLASS_NAMES, zero_division=0))

RF with targeted threshold for sulfonic acid only:
                 precision    recall  f1-score   support

carboxylic_acid       0.98      0.95      0.97       484
          amino       0.98      0.99      0.99       931
  sulfonic_acid       0.73      0.79      0.76        14
      guanidino       0.96      0.81      0.88       629

      micro avg       0.97      0.93      0.95      2058
      macro avg       0.91      0.89      0.90      2058
   weighted avg       0.97      0.93      0.95      2058
    samples avg       0.93      0.93      0.93      2058



Finding Optimal Threshold with Validation Set

In [ ]:
# 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

rf = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators = 100,
        class_weight = 'balanced',
        random_state = 42,
        n_jobs       = -1
    )
)
rf.fit(X_train_sc, y_train)

# Find optimal threshold on validation set
y_proba_val = np.array([est.predict_proba(X_val_sc)[:, 1]
                        for est in rf.estimators_]).T

thresholds_to_try = np.arange(0.05, 0.65, 0.05)

print("\nFinding optimal thresholds on validation set:")
optimal_thresholds = []

for i, class_name in enumerate(CLASS_NAMES):
    best_t, best_f1 = 0.5, 0
    for t in thresholds_to_try:
        preds = (y_proba_val[:, i] > t).astype(int)
        f1 = f1_score(y_val[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t  = t
    optimal_thresholds.append(best_t)
    print(f"  {class_name:<20}: threshold={best_t:.2f}  val F1={best_f1:.3f}")

# Apply optimal thresholds to test set
y_proba_test = np.array([est.predict_proba(X_test_sc)[:, 1]
                         for est in rf.estimators_]).T

y_pred_final = np.column_stack([
    (y_proba_test[:, i] > optimal_thresholds[i]).astype(int)
    for i in range(len(CLASS_NAMES))
])

print("\nFinal results on held-out test set:")
print(classification_report(y_test, y_pred_final,
    target_names=CLASS_NAMES, zero_division=0))
print(f"Macro F1: {f1_score(y_test, y_pred_final, average='macro', zero_division=0):.3f}")

Train: (5737, 1800)
Val:   (1229, 1800)
Test:  (1230, 1800)

Finding optimal thresholds on validation set:
  carboxylic_acid     : threshold=0.45  val F1=0.972
  amino               : threshold=0.60  val F1=0.990
  sulfonic_acid       : threshold=0.30  val F1=0.727
  guanidino           : threshold=0.50  val F1=0.873

Final results on held-out test set:
                 precision    recall  f1-score   support

carboxylic_acid       0.96      0.95      0.96       359
          amino       0.99      0.99      0.99       710
  sulfonic_acid       0.45      0.42      0.43        12
      guanidino       0.93      0.82      0.87       451

      micro avg       0.97      0.92      0.94      1532
      macro avg       0.84      0.79      0.81      1532
   weighted avg       0.96      0.92      0.94      1532
    samples avg       0.93      0.93      0.92      1532

Macro F1: 0.813
